In [ ]:
import pandas as pd
import ast
import json
import psycopg2
from psycopg2 import sql
import numpy as np 
import psycopg2.extras as extras
from rapidfuzz import process, fuzz

CONFIGURATION

In [2]:
from dotenv import load_dotenv, find_dotenv
import os
load_dotenv(find_dotenv())

CSV_FILE = os.getenv('CSV_FILE')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
TARGET_TABLE = os.getenv('TARGET_TABLE')

DB CONNECTION

In [3]:
#Establish Connection to PostgreSQL Database
try:
    conn = psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT
    )
    print(f"Successfully connected to PostgreSQL database: {DB_NAME}")

except psycopg2.Error as e:
    print(f"Database connection failed. Error: {e}")
    print("Please check your database service and credentials.")

Successfully connected to PostgreSQL database: EMF_Camp


CREATE TABLE

In [4]:
# Create the target table structure in PostgreSQL if it doesn't exist
create_table_query = sql.SQL(f"""
        CREATE TABLE IF NOT EXISTS {TARGET_TABLE}(
            id INTEGER UNIQUE,
            type VARCHAR(255),
            names VARCHAR(255),
            pronouns VARCHAR(100),
            title VARCHAR(255),
            description VARCHAR(255),
            short_description VARCHAR(255),
            video_privacy VARCHAR(100),
            is_fave BOOLEAN,
            official_content BOOLEAN,
            slug VARCHAR(255),
            link VARCHAR(255),
            occurrences VARCHAR(255),
            cost VARCHAR(100),
            equipment VARCHAR(255),
            age_range VARCHAR(100),
            drop_in BOOLEAN,
            family_friendly BOOLEAN,
            content_note VARCHAR(255),
            transcript VARCHAR(255), -- If I can find them
            load_timestamp TIMESTAMP WITH TIME ZONE DEFAULT CURRENT_TIMESTAMP
        );
    """
)


try:
    with conn.cursor() as cur:
        cur.execute(create_table_query)
        conn.commit()
    print(f"Table '{TARGET_TABLE}' already exists or has been successully created!")
except psycopg2.Error as e:
    print(f" Error creating table: {e}")
    conn.rollback()

Table 'etl_transformed_schedule' already exists or has been successully created!


EXTRACT (E) STEP

In [45]:
# Extract Data (E)
print("---Step 1: Extraction (E) ---")

def extract_data_from_csv(csv_file_path): 
    """
    Extracts data from a CSV file and returns a Pandas DataFrame.
    
    Parameters:
        csv_file_path (str): The path to the CSV file.
    """
    try:
        # Read the CSV file into a Pandas DataFrame
        df = pd.read_csv(csv_file_path, encoding="utf-8")
        print(f"Data extracted successfully from '{csv_file_path}'.")
        print(f"Initial DataFrame shape: {df.shape}")
        print("\nInitial Data Head:")
        display(df.head(1))
        return df
    except FileNotFoundError:
        print(f" Error: The file '{csv_file_path}' was not found. Make sure it's in the same directory.")
        df = None
    except Exception as e:
        print(f" Error during CSV read: {e}")
        df = None

    if df is None:
        raise SystemExit("Exiting ETL process due to extraction failure.")

# Robustly load csv to pandas
Schedule_CSV_PATH = "../../EMF_Camp/Raw_data/schedule.csv"
Transcripts_CSV_PATH = "../../EMF_Camp/Raw_data/transcripts.csv"
schedule_df = extract_data_from_csv(Schedule_CSV_PATH)
transcripts_df = extract_data_from_csv(Transcripts_CSV_PATH)

---Step 1: Extraction (E) ---
Data extracted successfully from '../../EMF_Camp/Raw_data/schedule.csv'.
Initial DataFrame shape: (550, 19)

Initial Data Head:


,id,type,names,pronouns,title,description,short_description,video_privacy,is_fave,official_content,slug,link,occurrences,cost,equipment,age_range,drop_in,family_friendly,content_note
0,384,performance,"Field-FX organisers, DJ Deathboy",NaN,Demoscene Livecode Jam,We are going to code exciting visual effects i...,Visual livecoding jam with a DJ,public,False,False,demoscene-livecode-jam,https://www.emfcamp.org/schedule/2026/384-demo...,"[{'occurrence_num': 1, 'start_date': '2026-07-...",NaN,NaN,NaN,NaN,NaN,NaN


Data extracted successfully from '../../EMF_Camp/Raw_data/transcripts.csv'.
Initial DataFrame shape: (129, 2)

Initial Data Head:


,title,transcript
0,3D Modelmaking ＂for girls＂ - The overlooked en...,[00:00] Hello.\n[00:17] Hello.\n[00:19] My nam...


TRANSFORM (T) STEP

In [63]:
# --- 2. TRANSFORMATION (T) ---
print("\n--- Step 2: Transformation (T) ---")

def fuzzy_match(title):
    """Fuzzy-matches transcript titles against schedule titles"""
    match = process.extractOne(
        title,
        transcript_titles,
        scorer=fuzz.WRatio,
        score_cutoff=90
    )
    if match:
        return match[0], match[1], match[2]
    return None, 0, None



def closest_schedule(title):
    """Find closest schedule title for unmatched transcripts"""
    match = process.extractOne(
        title,
        schedule_df["title"].dropna().tolist(),
        scorer=fuzz.WRatio
    )
    if match:
        row = schedule_df[schedule_df["title"] == match[0]].iloc[0]
        return match[0], row["type"], match[1]
    return None, None, 0

# Prepare transcript titles for matching
transcript_titles = transcripts_df["title"].dropna().tolist()


# --- DATA PREPERATION ---
# Separate talks and performances from other schedule entries. Testing and context indicates these are the only categories for transcripts
matchable_events = schedule_df[schedule_df["type"].isin(["talk", "performance"])].copy()
other_events = schedule_df[~schedule_df["type"].isin(["talk", "performance"])].copy()
print(f"Talks and performances: {len(matchable_events)}")

# Collect list of transcript titles for later troubleshooting
transcript_titles = transcripts_df["title"].dropna().tolist()


# --- MATCH TITLES ---
# Fuzzy-match schedule titles against transcript titles
matchable_events[["matched_title", "match_score", "match_index"]] = (
    matchable_events["title"].apply(
        lambda x: pd.Series(fuzzy_match(x))
    )
)

# --- ENRICH DATA ---
# Merge transcripts onto talks and performances
enriched_events = pd.merge(
    matchable_events,
    transcripts_df,
    left_on="matched_title",
    right_on="title",
    how="left",
    suffixes=("_schedule", "_transcript")
)


# --- IDENTIFY UNMATCHED TRANSCRIPTS ---
# Capture matched transcript titles BEFORE cleaning up the merged DataFrame
matched_titles = enriched_events["title_transcript"].dropna().unique()
# Find genuinely unmatched transcripts
unmatched_transcripts = transcripts_df[~transcripts_df["title"].isin(matched_titles)].copy()

# Find closest schedule title and event type for unmatched transcripts
results = unmatched_transcripts["title"].apply(closest_schedule).tolist()


# --- MATCHING STATS AND TROUBLESHOOTING ---
results_df = pd.DataFrame(
    results,
    columns=[
        "closest_schedule_title",
        "schedule_type",
        "similarity_score"
    ],
    index=unmatched_transcripts.index
)
unmatched_transcripts[["closest_schedule_title", "schedule_type", "similarity_score"]] = results_df

# Display unmatched transcripts for troubleshooting
print("\nDataframe of Transcripts where a coresponding event in schedule could not be found")
display(
    unmatched_transcripts[
        [
            "title",
            "closest_schedule_title",
            "schedule_type",
            "similarity_score"
        ]
    ].sort_values(
        "similarity_score",
        ascending=False
    )
)
matched_count = enriched_events["transcript"].notna().sum()
unmatched_count = len(unmatched_transcripts)
total_transcripts = len(transcripts_df)

print(f"Total transcripts: {total_transcripts}")
print(f"Matched transcripts: {matched_count}")
print(f"Unmatched transcripts: {unmatched_count}")
print(f"Match rate: {matched_count / total_transcripts * 100:.1f}%")


# --- CLEANUP AND RECOMBINE ---
# Remove temporary matching columns and duplicate transcript title
enriched_events = enriched_events.drop(
    columns=[
        "title_transcript",
        "matched_title",
        "match_score",
        "match_index"
    ],
    errors="ignore"
)
# Rename original schedule title back to title
enriched_events = enriched_events.rename(
    columns={"title_schedule": "title"}
)
# Recombine enriched talks and performances with other schedule entries
enriched_schedule = pd.concat(
    [enriched_events, other_events],
    ignore_index=True,
    sort=False
)

display(enriched_schedule.iloc[1])


--- Step 2: Transformation (T) ---
Talks and performances: 228

Dataframe of Transcripts where a coresponding event in schedule could not be found


,title,closest_schedule_title,schedule_type,similarity_score


Total transcripts: 129
Matched transcripts: 129
Unmatched transcripts: 0
Match rate: 100.0%


id                                                                  82
type                                                              talk
names                                                  James Singleton
pronouns                                                        he/him
title                       💚 Our Family's Electrotech Journey 🌞🏡🔌⛽️🚘️
description          The story of how we went all electric, ditched...
short_description    How we went all electric, ditched fossil fuels...
video_privacy                                                   public
is_fave                                                          False
official_content                                                  True
slug                                   our-familys-electrotech-journey
link                 https://www.emfcamp.org/schedule/2026/82-our-f...
occurrences          [{'occurrence_num': 1, 'start_date': '2026-07-...
cost                                                               NaN
equipm

# Transformation Expand to unique events and extract Datetime and relevent nested data from occurences WORK IN PROGRESS
- Anomolus behaviour and overeliance on vibecoding.  Return to this and check carefully. 
- Aim is to return the dataframe with occurences seperated into unique rows with the relevent nested data. multiple occurences Y/N flag for conviencence. 

In [98]:
# T2: IDENTIFY MULTIPLE OCCURRENCES AND EXPAND SCHEDULE


# ---------------------------------------------------------
# 1. PARSE OCCURRENCES
# ---------------------------------------------------------

def parse_occurrences(value, row_index):
    """CONVERT OCCURRENCES INTO A STANDARDISED LIST OF DICTIONARIES"""

    if value is None:
        return []

    if isinstance(value, str):

        value = value.strip()

        if not value:
            return []

        try:
            value = ast.literal_eval(value)

        except (ValueError, SyntaxError):

            try:
                value = json.loads(value)

            except (ValueError, TypeError):
                raise ValueError(
                    f"Could not parse occurrences at row {row_index}: {value}"
                )

    if not isinstance(value, (dict, list, tuple)):

        if pd.isna(value):
            return []

        raise TypeError(
            f"Unexpected occurrences format at row {row_index}: {type(value)}"
        )

    if isinstance(value, dict):
        occurrences = [value]

    else:
        occurrences = list(value)

    for occurrence in occurrences:

        if not isinstance(occurrence, dict):
            raise TypeError(
                f"Unexpected occurrence format at row {row_index}: {occurrence}"
            )

    return occurrences


# ---------------------------------------------------------
# 2. PARSE OCCURRENCES AND ADD MULIPLE OCCURANCES Y/N FLAG
# ---------------------------------------------------------
parsed_occurrences = [
    parse_occurrences(row["occurrences"], index)
    for index, row in enriched_schedule.iterrows()
]

enriched_schedule["multiple_occurrences"] = [
    "Y" if len(occurrences) > 1 else "N"
    for occurrences in parsed_occurrences
]

# ---------------------------------------------------------
# 3. EXPAND INTO INDIVIDUAL OCCURRENCE ROWS
# ---------------------------------------------------------
expanded_rows = []

for (row_index, row), occurrences in zip(
    enriched_schedule.iterrows(),
    parsed_occurrences
):

    # RETAIN ORIGINAL EVENT INFORMATION
    base_row = row.to_dict()

    # REMOVE ORIGINAL NESTED OCCURRENCES LIST FROM EXPANDED ROW
    base_row.pop("occurrences", None)

    # REMOVE PREVIOUSLY GENERATED DATE COLUMNS
    base_row.pop("start_date", None)
    base_row.pop("end_date", None)
    base_row.pop("occurrence_data", None)

    # RETAIN SOURCE REFERENCE
    base_row["source_row_index"] = row_index

    # HANDLE EVENTS WITHOUT OCCURRENCES
    if len(occurrences) == 0:

        expanded_row = base_row.copy()

        expanded_row.update({
            "occurrence_data": None,
            "occurrence_number": pd.NA,
            "occurrence_count": 0,
            "start_date": pd.NaT,
            "end_date": pd.NaT
        })

        expanded_rows.append(expanded_row)

    # CREATE ONE ROW PER INDIVIDUAL OCCURRENCE
    else:

        for occurrence_number, occurrence in enumerate(
            occurrences,
            start=1
        ):

            expanded_row = base_row.copy()

            expanded_row.update({

                # INDIVIDUAL OCCURRENCE DICTIONARY
                "occurrence_data": occurrence.copy(),

                # OCCURRENCE TRACKING
                "occurrence_number": occurrence_number,
                "occurrence_count": len(occurrences),

                # OCCURRENCE-SPECIFIC DATES
                "start_date": occurrence.get("start_date"),
                "end_date": occurrence.get("end_date")
            })

            expanded_rows.append(expanded_row)


# ---------------------------------------------------------
# 4. CREATE FINAL EXPANDED DATAFRAME
# ---------------------------------------------------------
expanded_schedule = pd.DataFrame(expanded_rows)

expanded_schedule["start_date"] = pd.to_datetime(
    expanded_schedule["start_date"],
    errors="raise"
)

expanded_schedule["end_date"] = pd.to_datetime(
    expanded_schedule["end_date"],
    errors="raise"
)

# ---------------------------------------------------------
# 5. CREATE MULTIPLE OCCURRENCES TROUBLESHOOTING DATAFRAME
# ---------------------------------------------------------

multiple_occurrences_df = expanded_schedule[
    expanded_schedule["multiple_occurrences"] == "Y"
].copy()


# ---------------------------------------------------------
# 6. TRANSFORMATION RESULTS
# ---------------------------------------------------------

print("\n--- OCCURRENCE EXPANSION RESULTS ---")
print(f"Original schedule rows: {len(enriched_schedule)}")

print(
    f"Original events with multiple occurrences: "
    f"{(enriched_schedule['multiple_occurrences'] == 'Y').sum()}"
)

print(f"Expanded schedule rows: {len(expanded_schedule)}")

print(
    f"Additional rows created: "
    f"{len(expanded_schedule) - len(enriched_schedule)}"
)

print(
    f"Individual rows requiring troubleshooting: "
    f"{len(multiple_occurrences_df)}"
)


# ---------------------------------------------------------
# 7. DATE COMPLETENESS
# ---------------------------------------------------------

print("\n--- EVENT DATE COMPLETENESS ---")

print(f"Rows with start dates: {expanded_schedule['start_date'].notna().sum()}")
print(f"Rows with end dates: {expanded_schedule['end_date'].notna().sum()}")

print(f"Missing start dates: {expanded_schedule['start_date'].isna().sum()}")
print(f"Missing end dates: {expanded_schedule['end_date'].isna().sum()}")


# ---------------------------------------------------------
# 8. DISPLAY FINAL EXPANDED DATA
# ---------------------------------------------------------

print("\n--- FINAL EXPANDED SCHEDULE ---")

display(
    expanded_schedule[
        [
            "id",
            "title",
            "multiple_occurrences",
            "occurrence_number",
            "occurrence_count",
            "start_date",
            "end_date",
            "occurrence_data"
        ]
    ].head(20)
)


# ---------------------------------------------------------
# 9. DISPLAY TROUBLESHOOTING DATA
# ---------------------------------------------------------

print("\n--- MULTIPLE OCCURRENCES TROUBLESHOOTING ---")

display(multiple_occurrences_df)


# ---------------------------------------------------------
# 10. FINAL VALIDATION
# ---------------------------------------------------------

assert "multiple_occurrences" in expanded_schedule.columns

assert "occurrence_data" in expanded_schedule.columns

assert pd.api.types.is_datetime64_any_dtype(
    expanded_schedule["start_date"]
)

assert pd.api.types.is_datetime64_any_dtype(
    expanded_schedule["end_date"]
)

assert (
    multiple_occurrences_df["multiple_occurrences"] == "Y"
).all()

print("\nT2 TRANSFORMATION COMPLETE.")


# EXPORT ALL COLUMN NAMES AND DTYPES WITHOUT TRUNCATION
dtype_report = "\n".join(
    f"{index + 1}. {column}: {dtype}"
    for index, (column, dtype) in enumerate(expanded_schedule.dtypes.items())
)
print(dtype_report)
with open("expanded_schedule_dtypes.txt", "w") as file:
    file.write(dtype_report)
print(f"\nTotal columns: {len(expanded_schedule.columns)}")
print("Complete dtype report saved to expanded_schedule_dtypes.txt")


--- OCCURRENCE EXPANSION RESULTS ---
Original schedule rows: 550
Original events with multiple occurrences: 16
Expanded schedule rows: 585
Additional rows created: 35
Individual rows requiring troubleshooting: 51

--- EVENT DATE COMPLETENESS ---
Rows with start dates: 583
Rows with end dates: 583
Missing start dates: 2
Missing end dates: 2

--- FINAL EXPANDED SCHEDULE ---


,id,title,multiple_occurrences,occurrence_number,occurrence_count,start_date,end_date,occurrence_data
0,384,Demoscene Livecode Jam,N,1,1,2026-07-17 19:30:00,2026-07-17 20:30:00,"{'occurrence_num': 1, 'start_date': '2026-07-1..."
1,82,💚 Our Family's Electrotech Journey 🌞🏡🔌⛽️🚘️,N,1,1,2026-07-18 12:40:00,2026-07-18 13:10:00,"{'occurrence_num': 1, 'start_date': '2026-07-1..."
2,98,So - how does the Internet really work,N,1,1,2026-07-19 12:40:00,2026-07-19 13:10:00,"{'occurrence_num': 1, 'start_date': '2026-07-1..."
3,431,The wonderful world of art toys,N,1,1,2026-07-18 18:00:00,2026-07-18 18:30:00,"{'occurrence_num': 1, 'start_date': '2026-07-1..."
4,157,The Machine That Says No: 1-Bit Music on the Z...,N,1,1,2026-07-18 12:00:00,2026-07-18 12:30:00,"{'occurrence_num': 1, 'start_date': '2026-07-1..."
5,203,Did a Robot Knit Your Jumper?,N,1,1,2026-07-19 14:40:00,2026-07-19 15:10:00,"{'occurrence_num': 1, 'start_date': '2026-07-1..."
6,204,"The automation wars... One marriage, two tech ...",N,1,1,2026-07-19 13:20:00,2026-07-19 13:50:00,"{'occurrence_num': 1, 'start_date': '2026-07-1..."
7,367,Disco Karaoke Rickshaw,N,1,1,2026-07-18 20:00:00,2026-07-18 23:00:00,"{'occurrence_num': 1, 'start_date': '2026-07-1..."
8,36,Signalling on the London Underground,N,1,1,2026-07-18 19:00:00,2026-07-18 19:40:00,"{'occurrence_num': 1, 'start_date': '2026-07-1..."
9,40,Temperature in space - how and why we make con...,N,1,1,2026-07-19 11:50:00,2026-07-19 12:20:00,"{'occurrence_num': 1, 'start_date': '2026-07-1..."



--- MULTIPLE OCCURRENCES TROUBLESHOOTING ---


,id,type,names,pronouns,title,description,short_description,video_privacy,is_fave,official_content,...,family_friendly,content_note,transcript,multiple_occurrences,source_row_index,occurrence_data,occurrence_number,occurrence_count,start_date,end_date
87,376,talk,Thor,NaN,Lightning Talks,This time is reserved for Lightning Talks - si...,NaN,public,False,True,...,False,NaN,[00:00] We have a lightning talk.\n[00:21] We ...,Y,87,"{'occurrence_num': 1, 'start_date': '2026-07-1...",1,2,2026-07-17 16:50:00,2026-07-17 17:50:00
88,376,talk,Thor,NaN,Lightning Talks,This time is reserved for Lightning Talks - si...,NaN,public,False,True,...,False,NaN,[00:00] We have a lightning talk.\n[00:21] We ...,Y,87,"{'occurrence_num': 2, 'start_date': '2026-07-1...",2,2,2026-07-18 19:50:00,2026-07-18 20:30:00
229,165,workshop,Zoë Hutber,she/her,Aardman Modelling Workshops (all ages),Make your own Aardman character to take home! ...,Build your own clay model of an Aardman charac...,none,False,True,...,True,NaN,NaN,Y,228,"{'occurrence_num': 1, 'start_date': '2026-07-1...",1,2,2026-07-18 13:00:00,2026-07-18 14:00:00
230,165,workshop,Zoë Hutber,she/her,Aardman Modelling Workshops (all ages),Make your own Aardman character to take home! ...,Build your own clay model of an Aardman charac...,none,False,True,...,True,NaN,NaN,Y,228,"{'occurrence_num': 2, 'start_date': '2026-07-1...",2,2,2026-07-18 15:00:00,2026-07-18 16:00:00
269,190,workshop,"Julie, Jon and Emily Spriggs","She/Her, He/Him, She/Her",Board Game Tent,"We bring board games, so you don't have to. Th...","We bring board games, or you bring board games...",public,False,True,...,True,NaN,NaN,Y,267,"{'occurrence_num': 1, 'start_date': '2026-07-1...",1,3,2026-07-18 15:00:00,2026-07-18 17:00:00
270,190,workshop,"Julie, Jon and Emily Spriggs","She/Her, He/Him, She/Her",Board Game Tent,"We bring board games, so you don't have to. Th...","We bring board games, or you bring board games...",public,False,True,...,True,NaN,NaN,Y,267,"{'occurrence_num': 2, 'start_date': '2026-07-1...",2,3,2026-07-18 18:30:00,2026-07-18 22:30:00
271,190,workshop,"Julie, Jon and Emily Spriggs","She/Her, He/Him, She/Her",Board Game Tent,"We bring board games, so you don't have to. Th...","We bring board games, or you bring board games...",public,False,True,...,True,NaN,NaN,Y,267,"{'occurrence_num': 3, 'start_date': '2026-07-1...",3,3,2026-07-19 15:00:00,2026-07-19 17:00:00
284,189,workshop,Louis Bougeard,he/him,Karaoke,Karaoke returns to EMF! Join us to sing along ...,Karaoke Returns!,public,False,False,...,False,Song choices and chosen queue names are modera...,NaN,Y,280,"{'occurrence_num': 1, 'start_date': '2026-07-1...",1,4,2026-07-16 19:30:00,2026-07-16 23:30:00
285,189,workshop,Louis Bougeard,he/him,Karaoke,Karaoke returns to EMF! Join us to sing along ...,Karaoke Returns!,public,False,False,...,False,Song choices and chosen queue names are modera...,NaN,Y,280,"{'occurrence_num': 2, 'start_date': '2026-07-1...",2,4,2026-07-17 19:30:00,2026-07-17 23:30:00
286,189,workshop,Louis Bougeard,he/him,Karaoke,Karaoke returns to EMF! Join us to sing along ...,Karaoke Returns!,public,False,False,...,False,Song choices and chosen queue names are modera...,NaN,Y,280,"{'occurrence_num': 3, 'start_date': '2026-07-1...",3,4,2026-07-18 20:30:00,2026-07-18 23:30:00



T2 TRANSFORMATION COMPLETE.
1. id: int64
2. type: str
3. names: str
4. pronouns: str
5. title: str
6. description: str
7. short_description: str
8. video_privacy: str
9. is_fave: bool
10. official_content: bool
11. slug: str
12. link: str
13. cost: str
14. equipment: str
15. age_range: str
16. drop_in: object
17. family_friendly: object
18. content_note: str
19. transcript: str
20. multiple_occurrences: str
21. source_row_index: int64
22. occurrence_data: object
23. occurrence_number: object
24. occurrence_count: int64
25. start_date: datetime64[us]
26. end_date: datetime64[us]

Total columns: 26
Complete dtype report saved to expanded_schedule_dtypes.txt


In [ ]:
# T3: Derived Column Creation (Unit Price)
# For this simple example, we'll assume every sale is for exactly 1 unit.
df['unit_price'] = df['sales_amount']
print("Derived column 'unit_price' created (assuming quantity=1).")

In [ ]:
# T4: Categorical Transformation (Creating a Sales Tier based on amount)

# 1. Define the list of conditions (must be Boolean arrays/Series)
conditions = [
    df['sales_amount'] >= 500,
    df['sales_amount'] >= 100
]

# 2. Define the list of values corresponding to the conditions
choices = [
    'High Value',
    'Medium Value'
]

# 3. Apply np.select: The "default" argument handles all cases that don't meet the conditions (i.e., less than 100)
df['sales_tier'] = np.select(conditions, choices, default='Low Value')

print("Categorical transformation 'sales_tier' created using np.select.")

In [ ]:
# T5: Filtering (Basic Data Quality Check)
# Remove rows where SalesAmount is less than or equal to 0
initial_rows = len(df)
df = df[df['sales_amount'] > 0]
rows_removed = initial_rows - len(df)
print(f" Data filtered: Removed {rows_removed} rows with non-positive sales amounts.")

LOAD (L)

In [ ]:
df_load = df[[
    'order_id', 'product', 'category', 'sales_amount', 'order_date',
    'region', 'customer_name', 'unit_price', 'sales_tier'
]]

In [ ]:
# Convert the DataFrame to a list of tuples for psycopg2's execute_batch
data_tuples = [tuple(row) for row in df_load.values]
# Define the INSERT query structure
cols = ', '.join(df_load.columns)
placeholders = ', '.join(['%s'] * len(df_load.columns))
# Define columns to update (all except the unique key 'order_id')
update_cols = [col for col in df_load.columns if col not in ['order_id']]
set_clause = ', '.join([f"{col} = EXCLUDED.{col}" for col in update_cols])
insert_query = sql.SQL(
    f"""
    INSERT INTO {TARGET_TABLE} ({cols}) 
    VALUES ({placeholders}) 
    ON CONFLICT (order_id) 
    DO UPDATE SET {set_clause}
    """
)
try:
    with conn.cursor() as cur:
        # Use execute_batch for efficient bulk insertion
        extras.execute_batch(cur, insert_query, data_tuples)
        conn.commit()
        print(f"Successfully attempted to load {len(data_tuples)} records into '{TARGET_TABLE}'. Conflicting records were updated.")

except psycopg2.Error as e:
    print(f"Error during data loading: {e}")
    conn.rollback()

finally:
    if conn:
        conn.close()
        print(" Database connection closed.")

print("\n--- ETL Process Complete ---")